In [2]:
# Step 1: Import required libraries and check dataset files

import os
import pandas as pd
import numpy as np

from sqlalchemy import create_engine
import matplotlib.pyplot as plt

# File paths
TRAIN_PATH = "/content/train.csv"
TEST_PATH = "/content/test.csv"
IDEAL_PATH = "/content/ideal.csv"

# Check whether files exist
for path in [TRAIN_PATH, TEST_PATH, IDEAL_PATH]:
    print(path, "exists:", os.path.exists(path))

# Load files
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
ideal_df = pd.read_csv(IDEAL_PATH)

# Show basic information
print("\nTraining data:")
print(train_df.head())
print(train_df.shape)

print("\nTest data:")
print(test_df.head())
print(test_df.shape)

print("\nIdeal functions:")
print(ideal_df.head())
print(ideal_df.shape)

/content/train.csv exists: True
/content/test.csv exists: True
/content/ideal.csv exists: True

Training data:
      x         y1         y2         y3        y4
0 -20.0  39.778572 -40.078590 -20.214268 -0.324914
1 -19.9  39.604813 -39.784000 -20.070950 -0.058820
2 -19.8  40.099070 -40.018845 -19.906782 -0.451830
3 -19.7  40.151100 -39.518402 -19.389118 -0.612044
4 -19.6  39.795662 -39.360065 -19.815890 -0.306076
(400, 5)

Test data:
      x          y
0  17.5  34.161040
1   0.3   1.215102
2  -8.7 -16.843908
3 -19.2 -37.170870
4 -11.0 -20.263054
(100, 2)

Ideal functions:
      x        y1        y2        y3        y4        y5        y6        y7  \
0 -20.0 -0.912945  0.408082  9.087055  5.408082 -9.087055  0.912945 -0.839071   
1 -19.9 -0.867644  0.497186  9.132356  5.497186 -9.132356  0.867644 -0.865213   
2 -19.8 -0.813674  0.581322  9.186326  5.581322 -9.186326  0.813674 -0.889191   
3 -19.7 -0.751573  0.659649  9.248426  5.659649 -9.248426  0.751573 -0.910947   
4 -19.6 -0.68196

SQLite database and load all CSV data

In [3]:
# Step 2: Create SQLite database and load datasets into tables

from sqlalchemy import create_engine

DB_PATH = "/content/python_assignment.db"
engine = create_engine(f"sqlite:///{DB_PATH}")

# Save dataframes to SQLite tables
train_df.to_sql("training_data", engine, if_exists="replace", index=False)
ideal_df.to_sql("ideal_functions", engine, if_exists="replace", index=False)
test_df.to_sql("test_data", engine, if_exists="replace", index=False)

# Check loaded tables
print("Database created:", DB_PATH)

print("\nTraining table:")
print(pd.read_sql("SELECT * FROM training_data LIMIT 5", engine))

print("\nIdeal functions table:")
print(pd.read_sql("SELECT * FROM ideal_functions LIMIT 5", engine))

print("\nTest table:")
print(pd.read_sql("SELECT * FROM test_data LIMIT 5", engine))

Database created: /content/python_assignment.db

Training table:
      x         y1         y2         y3        y4
0 -20.0  39.778572 -40.078590 -20.214268 -0.324914
1 -19.9  39.604813 -39.784000 -20.070950 -0.058820
2 -19.8  40.099070 -40.018845 -19.906782 -0.451830
3 -19.7  40.151100 -39.518402 -19.389118 -0.612044
4 -19.6  39.795662 -39.360065 -19.815890 -0.306076

Ideal functions table:
      x        y1        y2        y3        y4        y5        y6        y7  \
0 -20.0 -0.912945  0.408082  9.087055  5.408082 -9.087055  0.912945 -0.839071   
1 -19.9 -0.867644  0.497186  9.132356  5.497186 -9.132356  0.867644 -0.865213   
2 -19.8 -0.813674  0.581322  9.186326  5.581322 -9.186326  0.813674 -0.889191   
3 -19.7 -0.751573  0.659649  9.248426  5.659649 -9.248426  0.751573 -0.910947   
4 -19.6 -0.681964  0.731386  9.318036  5.731386 -9.318036  0.681964 -0.930426   

         y8        y9  ...        y41        y42       y43       y44  \
0 -0.850919  0.816164  ... -40.456474  40.2040

Select the best 4 ideal functions using least squares

In [4]:
# Step 3: Find best ideal function for each training function

def calculate_sse(actual_y, ideal_y):
    """
    Calculates the sum of squared errors between training y-values
    and ideal function y-values.
    """
    return np.sum((actual_y - ideal_y) ** 2)


training_columns = ["y1", "y2", "y3", "y4"]
ideal_columns = [col for col in ideal_df.columns if col != "x"]

selected_functions = []

for train_col in training_columns:
    best_ideal_col = None
    lowest_sse = float("inf")

    for ideal_col in ideal_columns:
        sse = calculate_sse(train_df[train_col], ideal_df[ideal_col])

        if sse < lowest_sse:
            lowest_sse = sse
            best_ideal_col = ideal_col

    max_deviation = np.max(np.abs(train_df[train_col] - ideal_df[best_ideal_col]))

    selected_functions.append({
        "training_function": train_col,
        "ideal_function": best_ideal_col,
        "least_square_error": lowest_sse,
        "max_deviation": max_deviation
    })

selected_df = pd.DataFrame(selected_functions)

print(selected_df)

  training_function ideal_function  least_square_error  max_deviation
0                y1            y42           34.246594       0.495968
1                y2            y41           35.601847       0.497703
2                y3            y11           29.861830       0.498936
3                y4            y48           31.963434       0.499742


Map test data to selected ideal functions

In [5]:
# Step 4: Map test data to selected ideal functions

import math

mapped_results = []

for _, test_row in test_df.iterrows():
    test_x = test_row["x"]
    test_y = test_row["y"]

    # Find matching x-value in ideal functions
    ideal_row = ideal_df[ideal_df["x"] == test_x]

    if ideal_row.empty:
        continue

    best_match = None
    smallest_deviation = float("inf")

    for selected in selected_functions:
        ideal_col = selected["ideal_function"]
        allowed_deviation = selected["max_deviation"] * math.sqrt(2)

        ideal_y = ideal_row.iloc[0][ideal_col]
        deviation = abs(test_y - ideal_y)

        if deviation <= allowed_deviation and deviation < smallest_deviation:
            smallest_deviation = deviation
            best_match = ideal_col

    if best_match is not None:
        mapped_results.append({
            "x": test_x,
            "y": test_y,
            "delta_y": smallest_deviation,
            "ideal_function": best_match
        })

mapped_df = pd.DataFrame(mapped_results)

print(mapped_df.head(20))
print("\nTotal mapped test points:", len(mapped_df))

       x          y   delta_y ideal_function
0   17.5  34.161040  0.351148            y41
1    0.3   1.215102  0.467342            y41
2    0.8   1.426456  0.532222            y41
3   14.0  -0.066506  0.134233            y48
4  -15.0  -0.205363  0.452371            y48
5    5.8  10.711373  0.656326            y41
6  -19.8 -19.915014  0.115014            y11
7   18.9  19.193245  0.293245            y11
8    8.8  -0.726051  0.488840            y48
9   -9.5  -9.652251  0.152251            y11
10   8.1 -16.659458  0.337686            y42
11  -8.8  16.571745  0.622709            y42
12  -3.1  -2.770136  0.329864            y11
13 -11.8  24.606413  0.646196            y42
14  18.8  37.523400  0.051833            y41
15   7.7  15.392297  0.501787            y41
16  -2.8  -3.298999  0.498999            y11
17  -8.2 -16.575344  0.295021            y41
18  14.1   0.310008  0.291442            y48
19   5.8  11.520408  0.152709            y41

Total mapped test points: 48


Save mapped results into SQLite table

In [6]:
# Step 5: Save mapped test results into SQLite database

mapped_df.to_sql("mapped_test_results", engine, if_exists="replace", index=False)

print("Mapped test results saved successfully.")

print(pd.read_sql("SELECT * FROM mapped_test_results LIMIT 10", engine))

print("\nDatabase tables:")
print(pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    engine
))

Mapped test results saved successfully.
      x          y   delta_y ideal_function
0  17.5  34.161040  0.351148            y41
1   0.3   1.215102  0.467342            y41
2   0.8   1.426456  0.532222            y41
3  14.0  -0.066506  0.134233            y48
4 -15.0  -0.205363  0.452371            y48
5   5.8  10.711373  0.656326            y41
6 -19.8 -19.915014  0.115014            y11
7  18.9  19.193245  0.293245            y11
8   8.8  -0.726051  0.488840            y48
9  -9.5  -9.652251  0.152251            y11

Database tables:
                  name
0        training_data
1      ideal_functions
2            test_data
3  mapped_test_results


Create Bokeh visualizations

In [7]:
# Step 6: Visualize training data, selected ideal functions, and mapped test data

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource
from bokeh.layouts import column

output_notebook()

plots = []

for selected in selected_functions:
    train_col = selected["training_function"]
    ideal_col = selected["ideal_function"]

    p = figure(
        title=f"Training Function {train_col} with Selected Ideal Function {ideal_col}",
        x_axis_label="X",
        y_axis_label="Y",
        width=850,
        height=350
    )

    p.circle(train_df["x"], train_df[train_col], size=4, legend_label=f"Training {train_col}")
    p.line(ideal_df["x"], ideal_df[ideal_col], line_width=2, legend_label=f"Ideal {ideal_col}")

    assigned = mapped_df[mapped_df["ideal_function"] == ideal_col]

    if not assigned.empty:
        p.triangle(
            assigned["x"],
            assigned["y"],
            size=7,
            legend_label="Mapped Test Points"
        )

    p.legend.location = "top_left"
    plots.append(p)

show(column(*plots))

Add custom exception and base class

In [9]:
# Step 7: Custom exception and base dataset class

class DatasetError(Exception):
    """
    Custom exception for dataset loading and validation errors.
    """
    pass


class BaseDataset:
    """
    Base class for loading and validating CSV datasets.
    """

    def __init__(self, file_path):
        """
        Initializes the dataset object.

        Parameters:
            file_path (str): Path of the CSV file.
        """
        self.file_path = file_path
        self.data = None

    def load_data(self):
        """
        Loads CSV data into a pandas DataFrame.
        """
        try:
            self.data = pd.read_csv(self.file_path)
            return self.data
        except FileNotFoundError:
            raise DatasetError(f"File not found: {self.file_path}")
        except pd.errors.EmptyDataError:
            raise DatasetError(f"File is empty: {self.file_path}")
        except Exception as e:
            raise DatasetError(f"Unexpected error while loading dataset: {e}")

    def validate_columns(self, required_columns):
        """
        Checks whether required columns exist in the dataset.

        Parameters:
            required_columns (list): Required column names.
        """
        if self.data is None:
            raise DatasetError("Dataset has not been loaded yet.")

        missing_columns = [
            col for col in required_columns if col not in self.data.columns
        ]

        if missing_columns:
            raise DatasetError(f"Missing columns: {missing_columns}")

        return True

Add inherited dataset classes

In [10]:
# Step 8: Child classes using inheritance

class TrainingDataset(BaseDataset):
    """
    Child class for the training dataset.
    """

    def validate(self):
        """
        Validates training dataset columns.
        """
        return self.validate_columns(["x", "y1", "y2", "y3", "y4"])


class TestDataset(BaseDataset):
    """
    Child class for the test dataset.
    """

    def validate(self):
        """
        Validates test dataset columns.
        """
        return self.validate_columns(["x", "y"])


class IdealDataset(BaseDataset):
    """
    Child class for the ideal functions dataset.
    """

    def validate(self):
        """
        Validates ideal dataset columns.
        """
        required_columns = ["x"] + [f"y{i}" for i in range(1, 51)]
        return self.validate_columns(required_columns)


# Test the inherited classes
training_dataset = TrainingDataset(TRAIN_PATH)
test_dataset = TestDataset(TEST_PATH)
ideal_dataset = IdealDataset(IDEAL_PATH)

train_df = training_dataset.load_data()
test_df = test_dataset.load_data()
ideal_df = ideal_dataset.load_data()

print("Training valid:", training_dataset.validate())
print("Test valid:", test_dataset.validate())
print("Ideal valid:", ideal_dataset.validate())

Training valid: True
Test valid: True
Ideal valid: True


Add database manager class

In [11]:
# Step 9: Database manager class

class DatabaseManager:
    """
    Handles SQLite database creation and table storage using SQLAlchemy.
    """

    def __init__(self, database_path):
        """
        Initializes the database manager.

        Parameters:
            database_path (str): Path of the SQLite database file.
        """
        self.database_path = database_path
        self.engine = create_engine(f"sqlite:///{database_path}")

    def save_dataframe(self, dataframe, table_name):
        """
        Saves a pandas DataFrame into a database table.
        """
        try:
            dataframe.to_sql(table_name, self.engine, if_exists="replace", index=False)
            return True
        except Exception as e:
            raise DatasetError(f"Could not save table {table_name}: {e}")

    def read_table(self, table_name):
        """
        Reads a database table into a pandas DataFrame.
        """
        try:
            return pd.read_sql(f"SELECT * FROM {table_name}", self.engine)
        except Exception as e:
            raise DatasetError(f"Could not read table {table_name}: {e}")


# Test database manager
database = DatabaseManager(DB_PATH)

database.save_dataframe(train_df, "training_data")
database.save_dataframe(ideal_df, "ideal_functions")
database.save_dataframe(test_df, "test_data")

print(database.read_table("training_data").head())

      x         y1         y2         y3        y4
0 -20.0  39.778572 -40.078590 -20.214268 -0.324914
1 -19.9  39.604813 -39.784000 -20.070950 -0.058820
2 -19.8  40.099070 -40.018845 -19.906782 -0.451830
3 -19.7  40.151100 -39.518402 -19.389118 -0.612044
4 -19.6  39.795662 -39.360065 -19.815890 -0.306076


Add ideal function selector class

In [12]:
# Step 10: Ideal function selector class

class IdealFunctionSelector:
    """
    Selects the best ideal functions for the four training functions
    using the least square method.
    """

    def __init__(self, training_data, ideal_data):
        self.training_data = training_data
        self.ideal_data = ideal_data
        self.selected_functions = []

    def calculate_sse(self, actual_y, ideal_y):
        """
        Calculates sum of squared errors.
        """
        return np.sum((actual_y - ideal_y) ** 2)

    def select_best_functions(self):
        """
        Selects one best ideal function for each training function.
        """
        training_columns = ["y1", "y2", "y3", "y4"]
        ideal_columns = [col for col in self.ideal_data.columns if col != "x"]

        for train_col in training_columns:
            best_ideal_col = None
            lowest_sse = float("inf")

            for ideal_col in ideal_columns:
                sse = self.calculate_sse(
                    self.training_data[train_col],
                    self.ideal_data[ideal_col]
                )

                if sse < lowest_sse:
                    lowest_sse = sse
                    best_ideal_col = ideal_col

            max_deviation = np.max(
                np.abs(self.training_data[train_col] - self.ideal_data[best_ideal_col])
            )

            self.selected_functions.append({
                "training_function": train_col,
                "ideal_function": best_ideal_col,
                "least_square_error": lowest_sse,
                "max_deviation": max_deviation
            })

        return pd.DataFrame(self.selected_functions)


selector = IdealFunctionSelector(train_df, ideal_df)
selected_df = selector.select_best_functions()
selected_functions = selector.selected_functions

print(selected_df)

  training_function ideal_function  least_square_error  max_deviation
0                y1            y42           34.246594       0.495968
1                y2            y41           35.601847       0.497703
2                y3            y11           29.861830       0.498936
3                y4            y48           31.963434       0.499742


Add test data mapper class

In [13]:
# Step 11: Test data mapper class

class TestDataMapper:
    """
    Maps test data points to selected ideal functions using the sqrt(2)
    deviation rule.
    """

    def __init__(self, test_data, ideal_data, selected_functions):
        self.test_data = test_data
        self.ideal_data = ideal_data
        self.selected_functions = selected_functions

    def map_test_data(self):
        """
        Maps each test x-y pair to one selected ideal function where possible.
        """
        mapped_results = []

        for _, test_row in self.test_data.iterrows():
            test_x = test_row["x"]
            test_y = test_row["y"]

            ideal_row = self.ideal_data[self.ideal_data["x"] == test_x]

            if ideal_row.empty:
                continue

            best_match = None
            smallest_deviation = float("inf")

            for selected in self.selected_functions:
                ideal_col = selected["ideal_function"]
                allowed_deviation = selected["max_deviation"] * math.sqrt(2)

                ideal_y = ideal_row.iloc[0][ideal_col]
                deviation = abs(test_y - ideal_y)

                if deviation <= allowed_deviation and deviation < smallest_deviation:
                    smallest_deviation = deviation
                    best_match = ideal_col

            if best_match is not None:
                mapped_results.append({
                    "x": test_x,
                    "y": test_y,
                    "delta_y": smallest_deviation,
                    "ideal_function": best_match
                })

        return pd.DataFrame(mapped_results)


mapper = TestDataMapper(test_df, ideal_df, selected_functions)
mapped_df = mapper.map_test_data()

print(mapped_df.head(20))
print("\nTotal mapped test points:", len(mapped_df))

       x          y   delta_y ideal_function
0   17.5  34.161040  0.351148            y41
1    0.3   1.215102  0.467342            y41
2    0.8   1.426456  0.532222            y41
3   14.0  -0.066506  0.134233            y48
4  -15.0  -0.205363  0.452371            y48
5    5.8  10.711373  0.656326            y41
6  -19.8 -19.915014  0.115014            y11
7   18.9  19.193245  0.293245            y11
8    8.8  -0.726051  0.488840            y48
9   -9.5  -9.652251  0.152251            y11
10   8.1 -16.659458  0.337686            y42
11  -8.8  16.571745  0.622709            y42
12  -3.1  -2.770136  0.329864            y11
13 -11.8  24.606413  0.646196            y42
14  18.8  37.523400  0.051833            y41
15   7.7  15.392297  0.501787            y41
16  -2.8  -3.298999  0.498999            y11
17  -8.2 -16.575344  0.295021            y41
18  14.1   0.310008  0.291442            y48
19   5.8  11.520408  0.152709            y41

Total mapped test points: 48


Save final mapped results with OOP database class

In [14]:
# Step 12: Save mapped results into database

database.save_dataframe(mapped_df, "mapped_test_results")

print("Mapped results saved successfully.")

print(database.read_table("mapped_test_results").head(10))

Mapped results saved successfully.
      x          y   delta_y ideal_function
0  17.5  34.161040  0.351148            y41
1   0.3   1.215102  0.467342            y41
2   0.8   1.426456  0.532222            y41
3  14.0  -0.066506  0.134233            y48
4 -15.0  -0.205363  0.452371            y48
5   5.8  10.711373  0.656326            y41
6 -19.8 -19.915014  0.115014            y11
7  18.9  19.193245  0.293245            y11
8   8.8  -0.726051  0.488840            y48
9  -9.5  -9.652251  0.152251            y11


Add visualization class

In [15]:
# Step 13: Visualization class using Bokeh

class DataVisualizer:
    """
    Creates Bokeh visualizations for training data, selected ideal functions,
    and mapped test data.
    """

    def __init__(self, training_data, ideal_data, mapped_data, selected_functions):
        self.training_data = training_data
        self.ideal_data = ideal_data
        self.mapped_data = mapped_data
        self.selected_functions = selected_functions

    def show_plots(self):
        """
        Displays training functions, selected ideal functions,
        and mapped test points.
        """
        output_notebook()
        plots = []

        for selected in self.selected_functions:
            train_col = selected["training_function"]
            ideal_col = selected["ideal_function"]

            p = figure(
                title=f"Training {train_col} and Ideal {ideal_col}",
                x_axis_label="X",
                y_axis_label="Y",
                width=850,
                height=350
            )

            p.circle(
                self.training_data["x"],
                self.training_data[train_col],
                size=4,
                legend_label=f"Training {train_col}"
            )

            p.line(
                self.ideal_data["x"],
                self.ideal_data[ideal_col],
                line_width=2,
                legend_label=f"Ideal {ideal_col}"
            )

            assigned = self.mapped_data[
                self.mapped_data["ideal_function"] == ideal_col
            ]

            if not assigned.empty:
                p.triangle(
                    assigned["x"],
                    assigned["y"],
                    size=7,
                    legend_label="Mapped test points"
                )

            p.legend.location = "top_left"
            plots.append(p)

        show(column(*plots))


visualizer = DataVisualizer(train_df, ideal_df, mapped_df, selected_functions)
visualizer.show_plots()

Unit Tests

In [16]:
# Step 14: Unit Tests

import unittest


class TestAssignment(unittest.TestCase):

    def test_training_dataset_loaded(self):
        self.assertEqual(train_df.shape, (400, 5))

    def test_test_dataset_loaded(self):
        self.assertEqual(test_df.shape, (100, 2))

    def test_ideal_dataset_loaded(self):
        self.assertEqual(ideal_df.shape, (400, 51))

    def test_selected_functions(self):
        self.assertEqual(len(selected_functions), 4)

    def test_mapping_dataframe(self):
        self.assertFalse(mapped_df.empty)

    def test_database_read(self):
        table = database.read_table("training_data")
        self.assertEqual(table.shape[0], 400)

    def test_sse(self):
        selector = IdealFunctionSelector(train_df, ideal_df)
        value = selector.calculate_sse(
            np.array([1, 2, 3]),
            np.array([1, 2, 3])
        )
        self.assertEqual(value, 0)


suite = unittest.TestLoader().loadTestsFromTestCase(TestAssignment)
unittest.TextTestRunner(verbosity=2).run(suite)

test_database_read (__main__.TestAssignment.test_database_read) ... ok
test_ideal_dataset_loaded (__main__.TestAssignment.test_ideal_dataset_loaded) ... ok
test_mapping_dataframe (__main__.TestAssignment.test_mapping_dataframe) ... ok
test_selected_functions (__main__.TestAssignment.test_selected_functions) ... ok
test_sse (__main__.TestAssignment.test_sse) ... ok
test_test_dataset_loaded (__main__.TestAssignment.test_test_dataset_loaded) ... ok
test_training_dataset_loaded (__main__.TestAssignment.test_training_dataset_loaded) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.055s

OK


<unittest.runner.TextTestResult run=7 errors=0 failures=0>